In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd

import os
import json
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

from scipy.io.wavfile import write


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1023)
           2	RESUME(arg=0, lineno=1023)
           4	LOAD_FAST(arg=0, lineno=1026)
           6	LOAD_CONST(arg=1, lineno=1026)
           8	BINARY_SUBSCR(arg=None, lineno=1026)
          12	LOAD_FAST(arg=0, lineno=1026)
          14	LOAD_CONST(arg=2, lineno=1026)
          16	BINARY_SUBSCR(arg=None, lineno=1026)
          20	COMPARE_OP(arg=68, lineno=1026)
          24	LOAD_FAST(arg=0, lineno=1026)
          26	LOAD_CONST(arg=1, lineno=1026)
          28	BINARY_SUBSCR(arg=None, lineno=1026)
          32	LOAD_FAST(arg=0, lineno=1026)
          34	LOAD_CONST(arg=3, lineno=1026)
          36	BINARY_SUBSCR(arg=None, lineno=1026)
          40	COMPARE_OP(arg=92, lineno=1026)
          44	BINARY_OP(arg=1, lineno=1026)
          48	RETURN_VALUE(arg=None, lineno=1026)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.

/venv/main/lib/python3.12/site-packages/phonemizer/utils.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## KSS

In [3]:
hps = utils.get_hparams_from_file("./configs/kss_base.json")

In [23]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("logs/kss_base/G_200000.pth", net_g, None)

In [28]:
stn_tst = get_text("저는 그림을 잘 못 그리지만 그림 그리는 걸 좋아해요.", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

In [29]:
stn_tst = get_text("그녀는 매일 아침 경쾌한 음악에 맞춰 운동을 한다.", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

In [30]:
stn_tst = get_text("몸이 안 좋았지만 산 정상까지 기를 쓰고 올라갔어요.", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

In [31]:
stn_tst = get_text("저는 지난달에 뇌 수술을 받았어요.", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

In [32]:
stn_tst = get_text("문제점을 보완한 후에 다시 논의합시다.", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))